In [ ]:
# Библиотека io в Python — это фундаментальный модуль для работы с
# потоками ввода-вывода (Input/Output). В отличие от простого вызова
# встроенной функции open(), работа через классы из io дает полный контроль над тем,
# как именно данные читаются, пишутся, буферизируются и кодируются
# В программировании «поток» — это абстрактный источник или приемник данных,
# который можно читать или записывать последовательно. Библиотека io предоставляет
# иерархию классов для работы с ними:

# IOBase: Корневой класс. Определяет базовый интерфейс (закрытие, контекстный менеджер).
# RawIOBase: Работа с «сырыми» (необработанными) байтами без буферизации. Прямое взаимодействие с ОС.
# BufferedIOBase: Добавляет буферизацию поверх сырых потоков для повышения производительности
# (чтение блоками, а не по байту).
# TextIOBase: Работа с текстом (str). Отвечает за перекодировку байтов в символы (используя encoding).

In [2]:
# io.StringIO — Файл в оперативной памяти (текст)

import io

# Создаем "файл" в памяти
output = io.StringIO()
print("Write all!!")
output.write("Привет, ")
output.write("мир!\n")
output.write(f"Текущее время: {__import__('datetime').datetime.now()}")

# Возвращаем курсор в начало
print("Read all!!")
output.seek(0)
print(output.read()) # Читаем все содержимое

output.close()

Write all!!
Read all!!
Привет, мир!
Текущее время: 2026-09-28 16:29:34.387132


In [3]:
# io.BytesIO — Файл в оперативной памяти (байты) 
# Аналог StringIO, но для бинарных данных (изображения, сжатые файлы, сериализованные объекты).

import io
import requests
from PIL import Image

response = requests.get("https://artchive.ru/res/media/img/ox800/work/57c/399752.jpg") 
response.raise_for_status() # Проверяем, что запрос прошел успешно

image_stream = io.BytesIO(response.content)
img = Image.open(image_stream)

img.show()

/usr/bin/eog: symbol lookup error: /snap/core20/current/lib/x86_64-linux-gnu/libpthread.so.0: undefined symbol: __libc_pthread_init, version GLIBC_PRIVATE


In [11]:
# 2. Фабрика экземпляра по классу (с сохранением конкретного типа)
# Зачем: без TypeVar пришлось бы возвращать object или использовать Any,
# и терялась бы конкретная типизация.

from typing import Type, TypeVar

T = TypeVar("T")

def make_instance(cls: Type[T]) -> T:
    return cls()

class Dog:
    def bark(self) -> None:
        print("woof")

dog: Dog = make_instance(Dog)
dog.bark()  # mypy знает, что это Dog, а не просто object
print(type(dog))

woof
<class '__main__.Dog'>


In [12]:
dd = Dog()
dd.bark()
print(type(dd))

woof
<class '__main__.Dog'>


In [13]:
# 3. Функция преобразования с сохранением типа контейнера
# Зачем: мы говорим анализатору: «в списке был тип T, после преобразования в
# нём всё ещё тип T». Это полезно, когда функция не меняет тип элементов.
from typing import List, TypeVar

T = TypeVar("T")

def map_list(items: List[T], fn) -> List[T]:
    return [fn(x) for x in items]

nums: List[int] = [1, 2, 3]
result: List[int] = map_list(nums, lambda x: x * 2)

In [14]:
# 4. Ограниченный TypeVar: только подтипы определённого класса
# Зачем: bound=Animal разрешает передавать любые подтипы Animal,
# но запрещает всё остальное. Это частый случай в Airflow/Pydantic-подобных
# конструкциях, где нужно работать с семейством классов.
from typing import TypeVar, List

class Animal:
    pass

class Dog(Animal):
    pass

class Cat(Animal):
    pass

T = TypeVar("T", bound=Animal)

def feed(animals: List[T]) -> None:
    for a in animals:
        # a гарантированно имеет интерфейс Animal
        ...

feed([Dog(), Cat()])      # OK
feed(["not an animal"])   # mypy: ошибка

In [ ]:
# Type[T] используют, когда нужно типизировать сам класс (а не его экземпляр) — например, передать класс как 
# аргумент или вернуть его из функции. Ниже 10 практических примеров с короткими пояснениями.
from typing import Type